In [0]:
dbutils.widgets.removeAll() 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "cat_ftr_smartdata_dev")
dbutils.widgets.text("source", "bronze")
dbutils.widgets.text("sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
schema_source = dbutils.widgets.get("source")
schema_sink = dbutils.widgets.get("sink")

In [0]:
df_producto = spark.table(f"{catalogo}.{schema_source}.productos")
df_venta = spark.table(f"{catalogo}.{schema_source}.ventas")

In [0]:
display(df_producto.limit(10))
display(df_venta.limit(10))

In [0]:
df_producto_select = df_producto.select("ID_ITEM", "PRODUCT", "CATEGORY")
#display(df_producto_select)

In [0]:
df_venta_valid = df_venta.filter("STATUS='VALID'")
df_venta_select = df_venta_valid.select("ID_ORDER", "STORE", "REG", "DATE", "ID_ITEM", "QTY", "PRICE", "DISCOUNT", "ID_SELLER")
#display(df_venta_select)


In [0]:
df_producto_alias = df_producto_select.alias("producto")
df_venta_alias = df_venta_select.alias("venta")

df_join = F.broadcast(df_producto_alias).join(
    df_venta_alias,
    F.col("producto.ID_ITEM") == F.col("venta.ID_ITEM"),
    "inner"
).select(
    F.col("producto.ID_ITEM").alias("ID_ARTICULO"),
    F.col("producto.PRODUCT").alias("PRODUCTO"),
    F.col("producto.CATEGORY").alias("CATEGORIA"),
    F.col("venta.ID_ORDER").alias("ID_PEDIDO"),
    F.col("venta.STORE").alias("TIENDA"),
    F.col("venta.REG").alias("NUM_LINEA"),
    F.col("venta.DATE").alias("FECHA"),
    F.col("venta.QTY").alias("CANTIDAD"),
    F.col("venta.PRICE").alias("PRECIO"),
    F.col("venta.DISCOUNT").alias("DESCUENTO"),
    F.col("venta.ID_SELLER").alias("ID_VENDEDOR")
)

#display(df_join)

In [0]:
# ─── Limpieza y Reglas de Calidad de Datos ─────────────────────────────

# 1. Normalizar campos de texto (trim + uppercase para consistencia)
df_clean = (
    df_join
    .withColumn("PRODUCTO",    F.trim(F.upper(F.col("PRODUCTO"))))
    .withColumn("CATEGORIA",   F.trim(F.upper(F.col("CATEGORIA"))))
    .withColumn("TIENDA",      F.trim(F.upper(F.col("TIENDA"))))
    .withColumn("ID_VENDEDOR", F.trim(F.col("ID_VENDEDOR")))
)

# 2. Eliminar registros con campos clave nulos
campos_clave = ["ID_ARTICULO", "ID_PEDIDO", "FECHA", "CANTIDAD", "PRECIO", "TIENDA"]
df_clean = df_clean.dropna(subset=campos_clave)

# 3. Filtrar valores fuera de rango lógico
df_clean = df_clean.filter(
    (F.col("CANTIDAD")  >  0) &
    (F.col("PRECIO")    >  0) &
    (F.col("DESCUENTO") >= 0) &
    (F.col("DESCUENTO") <= 100)
)

print("✓ Reglas de calidad aplicadas:")
print(f"   • Trim + UPPERCASE: PRODUCTO, CATEGORIA, TIENDA, ID_VENDEDOR")
print(f"   • Nulos eliminados en campos clave: {campos_clave}")
print(f"   • Filtros de rango: CANTIDAD > 0 | PRECIO > 0 | DESCUENTO en [0, 100]")

In [0]:
# ─── Transformaciones y Enriquecimiento para Capa Silver ────────────────

df_silver = (
    df_clean
    # ── Etiqueta de descuento (expresión nativa, sin UDF) ──
    .withColumn("ETIQUETA_DESCUENTO",
        F.when(F.col("DESCUENTO") == 0,   F.lit("Precio Normal"))
         .when(F.col("DESCUENTO") == 100, F.lit("Regalo"))
         .when((F.col("DESCUENTO") > 75) & (F.col("DESCUENTO") < 100), F.lit("Bono Empresarial"))
         .when((F.col("DESCUENTO") >  1) & (F.col("DESCUENTO") <= 74), F.lit("Promoción"))
         .otherwise(F.lit("Sin Clasificar"))
    )
    # ── Importes financieros ──
    .withColumn("IMPORTE_BRUTO",
        F.round(F.col("CANTIDAD") * F.col("PRECIO"), 2))
    .withColumn("IMPORTE_DESCUENTO",
        F.round(F.col("CANTIDAD") * F.col("PRECIO") * F.col("DESCUENTO") / 100, 2))
    .withColumn("IMPORTE_NETO",
        F.round(F.col("CANTIDAD") * F.col("PRECIO") * (1 - F.col("DESCUENTO") / 100), 2))
    # ── Dimensiones temporales ──
    .withColumn("ANIO",       F.year("FECHA"))
    .withColumn("MES",        F.month("FECHA"))
    .withColumn("TRIMESTRE",  F.quarter("FECHA"))
    .withColumn("DIA_SEMANA", F.dayofweek("FECHA"))   # 1=Dom, 7=Sáb
    .withColumn("ES_FIN_SEMANA",
        F.dayofweek("FECHA").isin(1, 7))
    # ── Segmento de precio unitario ──
    .withColumn("RANGO_PRECIO",
        F.when(F.col("PRECIO") <  10,  F.lit("Económico"))
         .when(F.col("PRECIO") <  50,  F.lit("Estándar"))
         .when(F.col("PRECIO") < 200,  F.lit("Premium"))
         .otherwise(F.lit("Lujo"))
    )
)

display(df_silver.limit(10))

In [0]:
def estado(descuento):
    if descuento == 0:
        return "Precio Normal"
    elif descuento == 100:
        return "Regalo"
    elif descuento > 75 and descuento < 100:
        return "bono empresarial"
    elif descuento > 1 and descuento <= 74:
        return "promoción"
    else:
        return None

precio_descuento_udf = F.udf(estado, StringType())

df_join_etiqueta = df_join.withColumn("ETIQUETA_DESCUENTO", precio_descuento_udf(F.col("DESCUENTO")))
display(df_join_etiqueta)

In [0]:
target_table = f"{catalogo}.{schema_sink}.ventas_productos_categorias"

(
    df_silver
    .select(
        F.col("ID_ARTICULO").cast("string"),
        F.col("PRODUCTO").cast("string"),
        F.col("CATEGORIA").cast("string"),
        F.col("ID_PEDIDO").cast("string"),
        F.col("TIENDA").cast("string"),
        F.col("NUM_LINEA").cast("string"),
        F.col("FECHA").cast("date"),
        F.col("CANTIDAD").cast("int"),
        F.col("PRECIO").cast("double"),
        F.col("DESCUENTO").cast("int"),
        F.col("ID_VENDEDOR").cast("string"),
        F.col("ETIQUETA_DESCUENTO").cast("string"),
        F.col("IMPORTE_BRUTO").cast("double"),
        F.col("IMPORTE_DESCUENTO").cast("double"),
        F.col("IMPORTE_NETO").cast("double"),
        F.col("ANIO").cast("int"),
        F.col("MES").cast("int"),
        F.col("TRIMESTRE").cast("int"),
        F.col("DIA_SEMANA").cast("int"),
        F.col("ES_FIN_SEMANA").cast("boolean"),
        F.col("RANGO_PRECIO").cast("string"),
    )
    .coalesce(4)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"✓ Tabla '{target_table}' escrita correctamente en capa Silver.")

In [0]:
%sql
select * from cat_ftr_smartdata_dev.silver.ventas_productos_categorias limit 10